## 0. Installation and imports

In [ ]:
import subprocess, sys

try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.12'])

try:
    from pytorch_grad_cam import GradCAM
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'grad-cam'])

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Ambiente: {"Google Colab" if IN_COLAB else "Local"}')

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cv2

import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import timm

from PIL import Image
from sklearn.metrics import f1_score

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

## 1. Configuration

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Trilha3-V2')
    IMGS_DIR = Path('/content/Imgs')
    import os
    if not os.path.exists('/content/Imgs'):
        print('Descompactando imagens...')
        import subprocess
        subprocess.run(['unzip', '-q',
                        '/content/drive/MyDrive/Trilha3-V2/Data/Imgs.zip',
                        '-d', '/content/'], check=True)
        print('Descompactação concluída!')
else:
    BASE_DIR = Path('e:/Endo-ICTAI-2026')
    IMGS_DIR = Path('e:/Endo-ICTAI-2026/Data/Imgs')

SPLITS_DIR  = BASE_DIR / 'splits'
CKPT_DIR    = BASE_DIR / 'checkpoints'
RESULTS_DIR = BASE_DIR / 'results' / 'nb05'
FIGS_DIR    = BASE_DIR / 'figs' / 'gradcam'
IMGS_OUT    = RESULTS_DIR / 'images'

for d in [RESULTS_DIR, FIGS_DIR, IMGS_OUT]:
    d.mkdir(parents=True, exist_ok=True)

CORE_LABELS = ['ENANTEMA', 'PÓLIPO', 'ÚLCERA', 'EROSÃO', 'MICRONODULARIDADE']
N_CLASSES   = len(CORE_LABELS)
N_FOLDS     = 5
IMG_SIZE    = 224
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

EXP_NAME     = 'm2_l06'
CKPT_PATTERN = f'swin_{EXP_NAME}_fold{{fold}}.pt'

if EXP_NAME in ('m0_swin', 'm2_l03', 'm2_l04', 'm2_l05', 'm2_l06', 'm2_l07'):
    RESULTS_JSON = BASE_DIR / 'results' / 'm2' / 'm2_results.json'
else:
    RESULTS_JSON = BASE_DIR / 'results' / 'nb04' / 'nb04_results.json'

N_IMGS_PER_CLASS = 5

print(f'Device      : {DEVICE}')
print(f'Experimento : {EXP_NAME}')
print(f'Checkpoint  : {CKPT_PATTERN}')
print(f'Results JSON: {RESULTS_JSON}')
print(f'Outputs     : {RESULTS_DIR}')

## 2. Dataset and model M2 λ=0.6

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

val_transform = T.Compose([
    T.Resize((int(IMG_SIZE * 1.14), int(IMG_SIZE * 1.14))),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class GastroscopyDataset(Dataset):
    def __init__(self, df, imgs_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.imgs_dir  = imgs_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img    = Image.open(self.imgs_dir / row['image_name']).convert('RGB')
        labels = torch.tensor(row[CORE_LABELS].values.astype(float), dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
        return img, labels

def build_swin_tiny(n_classes=N_CLASSES):
    return timm.create_model(
        'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k',
        pretrained=False,
        num_classes=n_classes
    )

def load_fold_model(fold: int) -> nn.Module:
    ckpt_path = CKPT_DIR / CKPT_PATTERN.format(fold=fold)
    assert ckpt_path.exists(), f'Checkpoint não encontrado: {ckpt_path}'
    model = build_swin_tiny()
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.to(DEVICE).eval()
    return model

print('Dataset e funções de modelo definidos.')

## 3. Grad-CAM for Swin-Tiny

Swin-Tiny does not have classical conv layers.  
The correct target for Grad-CAM is the **last block of the last stage** (`layers[-1].blocks[-1].norm1`).  
We use `reshape_transform` to convert the Swin patch tokens into a spatial grid.

In [ ]:
def swin_reshape_transform(tensor, height=7, width=7):
    """Converte tokens para (B, C, H, W) para o Grad-CAM."""
    if tensor.dim() == 3:
        result = tensor.reshape(tensor.size(0), height, width, tensor.size(2))
        return result.permute(0, 3, 1, 2)
    elif tensor.dim() == 4:
        return tensor.permute(0, 3, 1, 2)
    return tensor

def get_gradcam_for_label(model, img_tensor: torch.Tensor,
                          label_idx: int) -> np.ndarray:
    """
    Gera mapa Grad-CAM para um rótulo específico (label_idx 0-4).
    Retorna array (H, W) normalizado [0, 1].
    """
    target_layer = model.layers[-1].blocks[-1].norm1

    cam = GradCAM(
        model=model,
        target_layers=[target_layer],
        reshape_transform=swin_reshape_transform,
    )

    targets = [ClassifierOutputTarget(label_idx)]
    grayscale_cam = cam(input_tensor=img_tensor.unsqueeze(0).to(DEVICE),
                        targets=targets)
    return grayscale_cam[0]

def make_overlay(img_pil: Image.Image,
                 cam_map: np.ndarray,
                 alpha: float = 0.45) -> np.ndarray:
    """Sobrepõe heatmap Jet na imagem original. Retorna uint8 RGB."""
    import cv2
    import numpy as np
    from pytorch_grad_cam.utils.image import show_cam_on_image
    img_np = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.0
    overlay = show_cam_on_image(img_np, cam_map, use_rgb=True, colormap=cv2.COLORMAP_JET,
                                 image_weight=1 - alpha)
    return overlay

def cam_stats(cam_map: np.ndarray) -> dict:
    """Calcula estatísticas do mapa de ativação para o artigo."""
    import numpy as np
    flat = cam_map.flatten()
    top10 = float(np.mean(flat >= np.percentile(flat, 90)))
    top25 = float(np.mean(flat >= np.percentile(flat, 75)))
    top50 = float(np.mean(flat >= np.percentile(flat, 50)))

    pct_above_025 = float(np.mean(flat >= 0.25))
    pct_above_050 = float(np.mean(flat >= 0.50))
    pct_above_075 = float(np.mean(flat >= 0.75))

    peak_idx = np.unravel_index(np.argmax(cam_map), cam_map.shape)
    return {
        'mean':     float(cam_map.mean()),
        'std':      float(cam_map.std()),
        'max':      float(cam_map.max()),
        'top10_pct': top10,
        'top25_pct': top25,
        'top50_pct': top50,
        'pct_above_025': pct_above_025,
        'pct_above_050': pct_above_050,
        'pct_above_075': pct_above_075,
        'peak_row': int(peak_idx[0]),
        'peak_col': int(peak_idx[1]),
    }

print('Funções Grad-CAM definidas.')

## 4. Automatic interpretive text

Generates clinical description of the heatmap — for the paper and for the medical front-end.

In [ ]:
LABEL_PT = {
    'ENANTEMA':         'enantema (hiperemia mucosa)',
    'PÓLIPO':           'pólipo gástrico',
    'ÚLCERA':           'úlcera gástrica',
    'EROSÃO':           'erosão mucosa',
    'MICRONODULARIDADE':'micronodularidade',
}

LABEL_EXPECTED_REGION = {
    'ENANTEMA':         'difusa (padrão esperado para hiperemia generalizada)',
    'PÓLIPO':           'focal, correspondendo à estrutura elevada',
    'ÚLCERA':           'focal, correspondendo à cratera ulcerosa',
    'EROSÃO':           'focal a semifocal, correspondendo à lesão mucosa superficial',
    'MICRONODULARIDADE':'difusa a semifocal, correspondendo ao padrão nodular',
}

def localize_region(peak_row: int, peak_col: int,
                    h: int = 7, w: int = 7) -> str:
    """Descreve em linguagem natural a região de pico do heatmap."""
    vert  = 'superior' if peak_row < h // 3 else ('central' if peak_row < 2 * h // 3 else 'inferior')
    horiz = 'esquerda' if peak_col < w // 3 else ('central' if peak_col < 2 * w // 3 else 'direita')
    if vert == 'central' and horiz == 'central':
        return 'central'
    return f'{vert}-{horiz}'

def concentration_level(pct_above_050: float) -> str:
    """
    Classifica concentração usando limiar fixo pct_above_050
    (fração da imagem com ativação > 0.50).
      < 10%  → altamente concentrada (focal)
      10-30% → moderadamente concentrada
      > 30%  → difusa
    """
    if pct_above_050 < 0.10:
        return 'altamente concentrada (focal)'
    elif pct_above_050 < 0.30:
        return 'moderadamente concentrada'
    else:
        return 'difusa'

def generate_interpretation(label: str, stats: dict,
                            confidence: float,
                            has_luz: bool = False,
                            luz_overlap_pct: float = None) -> str:
    """
    Gera texto interpretativo automático do Grad-CAM.
    Inclui análise do shortcut LUZ→EROSÃO quando aplicável.
    """
    label_name = LABEL_PT.get(label, label)
    region     = localize_region(stats['peak_row'], stats['peak_col'])
    conc       = concentration_level(stats['pct_above_050'])
    expected   = LABEL_EXPECTED_REGION.get(label, 'região de interesse')

    lines = [
        f"Detecção de {label_name} com confiança {confidence:.1%}.",
        f"A atenção do modelo foi {conc}, concentrada na região {region} da imagem"
        f" (Grad-CAM mean={stats['mean']:.3f}, std={stats['std']:.3f}).",
        f"O padrão de ativação esperado para este achado é {expected}.",
        f"Fração da imagem com ativação >0.25: {stats['pct_above_025']*100:.1f}%; "
        f">0.50: {stats['pct_above_050']*100:.1f}%; "
        f">0.75: {stats['pct_above_075']*100:.1f}%.",
    ]

    if label == 'EROSÃO' and has_luz:
        if luz_overlap_pct is not None:
            if luz_overlap_pct > 0.30:
                lines.append(
                    f"ATENÇÃO — Sobreposição com região de reflexo de luz: {luz_overlap_pct:.1%}. "
                    f"Potencial shortcut LUZ→EROSÃO detectado nesta imagem. "
                    f"Validação médica recomendada: a ativação corresponde à lesão mucosa ou ao reflexo?"
                )
            else:
                lines.append(
                    f"Sobreposição com região de reflexo de luz: {luz_overlap_pct:.1%} (baixa). "
                    f"O modelo aparentemente focou na lesão mucosa, não no artefato de reflexo."
                )
        else:
            lines.append(
                "Esta imagem contém reflexo de luz (LUZ=1). "
                "Validação médica recomendada para confirmar se a ativação corresponde à lesão."
            )

    return ' '.join(lines)

print('Funções de interpretação definidas.')

## 5. Selection of images for analysis

For each class: selects images from the **test** set (fold 0) where:
- ground truth = 1 for the class
- model prediction = 1 (hit)
- ordered by decreasing confidence → top-N_IMGS_PER_CLASS

In [ ]:
df_te = pd.read_csv(SPLITS_DIR / 'fold_0_test.csv')
model_fold0 = load_fold_model(fold=0)

print(f'Imagens no fold-0 test: {len(df_te)}')

ds_te = GastroscopyDataset(df_te, IMGS_DIR, val_transform)
dl_te = DataLoader(ds_te, batch_size=32, shuffle=False, num_workers=0)

all_probs = []
with torch.no_grad():
    for imgs, _ in dl_te:
        logits = model_fold0(imgs.to(DEVICE))
        all_probs.append(torch.sigmoid(logits).cpu().numpy())

probs_arr = np.concatenate(all_probs)
print(f'Probabilidades obtidas: shape={probs_arr.shape}')

with open(RESULTS_JSON, encoding='utf-8') as f:
    exp_results = json.load(f)

val_thr_fold0 = np.array(exp_results[EXP_NAME]['fold_results'][0]['val_thresholds'])
print(f'Thresholds fold-0: {dict(zip(CORE_LABELS, val_thr_fold0.round(3)))}')

preds_arr  = (probs_arr >= val_thr_fold0).astype(int)
labels_arr = df_te[CORE_LABELS].values

f1_fold0 = f1_score(labels_arr, preds_arr, average='macro', zero_division=0)
print(f'F1-macro fold-0 test: {f1_fold0:.4f}')

In [ ]:
selected = {}

for i, label in enumerate(CORE_LABELS):

    correct_mask = (labels_arr[:, i] == 1) & (preds_arr[:, i] == 1)
    correct_idx  = np.where(correct_mask)[0]

    if len(correct_idx) == 0:
        print(f'  {label}: NENHUM acerto positivo no fold-0 test!')

        correct_idx = np.where(labels_arr[:, i] == 1)[0]

    conf_sorted = sorted(correct_idx, key=lambda idx: probs_arr[idx, i], reverse=True)
    top_idx     = conf_sorted[:N_IMGS_PER_CLASS]

    selected[label] = []
    for df_idx in top_idx:
        row = df_te.iloc[df_idx]
        selected[label].append({
            'df_idx':     int(df_idx),
            'image_name': row['image_name'],
            'confidence': float(probs_arr[df_idx, i]),
            'gt_label':   int(labels_arr[df_idx, i]),
            'pred_label': int(preds_arr[df_idx, i]),
            'has_luz':    int(row.get('LUZ', 0)),
            'label_idx':  i,
            'label_name': label,
        })

    confs = [s['confidence'] for s in selected[label]]
    print(f'  {label}: {len(selected[label])} imagens '
          f'(conf: {min(confs):.3f}–{max(confs):.3f})')

print(f'\nTotal de imagens selecionadas: {sum(len(v) for v in selected.values())}')

## 6. Generation of Grad-CAM maps and overlays

In [ ]:
all_metadata = []

for label, cases in selected.items():
    print(f'\nProcessando {label} ({len(cases)} imagens)...')

    for case in cases:
        img_name  = case['image_name']
        label_idx = case['label_idx']
        conf      = case['confidence']
        has_luz   = bool(case['has_luz'])

        img_pil = Image.open(IMGS_DIR / img_name).convert('RGB')

        img_tensor = val_transform(img_pil)

        cam_map = get_gradcam_for_label(model_fold0, img_tensor, label_idx)
        stats   = cam_stats(cam_map)

        luz_overlap = None
        if has_luz and label == 'EROSÃO':
            img_gray   = np.array(img_pil.convert('L').resize((cam_map.shape[1], cam_map.shape[0])))
            brilho_thr = np.percentile(img_gray, 95)
            luz_mask_img = img_gray >= brilho_thr

            if luz_mask_img.sum() > 0:
                luz_overlap = float(
                    cam_map[luz_mask_img].mean() / (cam_map.mean() + 1e-8)
                )

        interpretation = generate_interpretation(
            label, stats, conf,
            has_luz=has_luz, luz_overlap_pct=luz_overlap
        )

        overlay_rgb = make_overlay(img_pil, cam_map)

        stem         = Path(img_name).stem
        label_slug   = label.replace('Ó', 'O').replace('Ú', 'U').replace('Ã', 'A').replace('É', 'E')
        orig_path    = IMGS_OUT / f'{stem}_{label_slug}_original.png'
        overlay_path = IMGS_OUT / f'{stem}_{label_slug}_overlay.png'
        heatmap_path = IMGS_OUT / f'{stem}_{label_slug}_heatmap.png'

        img_pil.resize((IMG_SIZE, IMG_SIZE)).save(orig_path)
        Image.fromarray(overlay_rgb).save(overlay_path)

        heatmap_uint8 = (cam_map * 255).astype(np.uint8)
        heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
        heatmap_rgb   = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
        Image.fromarray(heatmap_rgb).save(heatmap_path)

        meta_entry = {
            **case,
            'cam_stats':       stats,
            'luz_overlap_pct': luz_overlap,
            'interpretation':  interpretation,
            'orig_path':       str(orig_path.name),
            'overlay_path':    str(overlay_path.name),
            'heatmap_path':    str(heatmap_path.name),
        }
        all_metadata.append(meta_entry)
        print(f'  {img_name} [{label}] conf={conf:.3f} → OK')

with open(RESULTS_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(all_metadata, f, ensure_ascii=False, indent=2)

print(f'\nMetadata salvo: {len(all_metadata)} entradas')

## 7. Analysis of the LIGHT→EROSION shortcut

In [ ]:
erosao_idx  = CORE_LABELS.index('EROSÃO')
erosao_mask = labels_arr[:, erosao_idx] == 1
luz_mask    = df_te['LUZ'].values == 1

n_erosao         = int(erosao_mask.sum())
n_erosao_com_luz = int((erosao_mask & luz_mask).sum())
n_erosao_sem_luz = int((erosao_mask & ~luz_mask).sum())

if luz_mask.sum() > 0:
    f1_erosao_com_luz = float(f1_score(
        labels_arr[luz_mask, erosao_idx],
        preds_arr [luz_mask, erosao_idx],
        average='binary', zero_division=0
    ))
else:
    f1_erosao_com_luz = float('nan')

no_luz_mask = ~luz_mask
if no_luz_mask.sum() > 0:
    f1_erosao_sem_luz = float(f1_score(
        labels_arr[no_luz_mask, erosao_idx],
        preds_arr [no_luz_mask, erosao_idx],
        average='binary', zero_division=0
    ))
else:
    f1_erosao_sem_luz = float('nan')

luz_sem_erosao_mask = luz_mask & ~erosao_mask
n_luz_sem_erosao    = int(luz_sem_erosao_mask.sum())
if n_luz_sem_erosao > 0:
    fp_rate_luz = float(preds_arr[luz_sem_erosao_mask, erosao_idx].mean())
else:
    fp_rate_luz = float('nan')

shortcut_stats = {
    'fold': 0,
    'n_erosao_total':       n_erosao,
    'n_erosao_com_luz':     n_erosao_com_luz,
    'n_erosao_sem_luz':     n_erosao_sem_luz,
    'cooc_pct':             float(n_erosao_com_luz / n_erosao) if n_erosao > 0 else 0.0,
    'f1_erosao_com_luz':    f1_erosao_com_luz,
    'f1_erosao_sem_luz':    f1_erosao_sem_luz,
    'n_luz_sem_erosao':     n_luz_sem_erosao,
    'fp_rate_luz':          fp_rate_luz,
    'cam_luz_overlaps':     [],
}

for entry in all_metadata:
    if entry['label_name'] == 'EROSÃO' and entry['has_luz'] and entry['luz_overlap_pct'] is not None:
        shortcut_stats['cam_luz_overlaps'].append({
            'image':       entry['image_name'],
            'overlap_pct': entry['luz_overlap_pct'],
            'confidence':  entry['confidence'],
        })

with open(RESULTS_DIR / 'shortcut_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(shortcut_stats, f, ensure_ascii=False, indent=2)

print('=== ANÁLISE DO SHORTCUT LUZ→EROSÃO ===')
print(f'Imagens EROSÃO no fold-0 test      : {n_erosao}')
print(f'  Com LUZ                          : {n_erosao_com_luz} ({n_erosao_com_luz/max(n_erosao,1):.1%})')
print(f'  Sem LUZ                          : {n_erosao_sem_luz}')
print(f'F1 EROSÃO (imagens COM LUZ, GT∈{{0,1}}): {f1_erosao_com_luz:.4f}')
print(f'F1 EROSÃO (imagens SEM LUZ)        : {f1_erosao_sem_luz:.4f}')
print(f'Imagens LUZ=1 mas EROSÃO_GT=0      : {n_luz_sem_erosao}')
print(f'Taxa FP EROSÃO nesses casos        : {fp_rate_luz:.1%}')
print()
if not (isinstance(f1_erosao_com_luz, float) and np.isnan(f1_erosao_com_luz)):
    delta = f1_erosao_com_luz - f1_erosao_sem_luz
    print(f'Δ F1 (com LUZ − sem LUZ)           : {delta:+.4f}')
    if delta > 0.05:
        print('  → F1 maior com LUZ: possível shortcut (modelo aproveita luz como sinal).')
    elif delta < -0.05:
        print('  → F1 menor com LUZ: luz confunde o modelo.')
    else:
        print('  → Sem diferença relevante: LUZ não impacta o F1 de EROSÃO.')
print()
if not np.isnan(fp_rate_luz):
    if fp_rate_luz > 0.20:
        print('  ⚠️  FP rate > 20%: shortcut LUZ→EROSÃO confirmado quantitativamente.')
    else:
        print('  ✓  FP rate ≤ 20%: shortcut não dominante nas predições.')

## 8. Figure for the paper — Grad-CAM Grid

Format: 5 rows (one per class) × 3 columns (original / heatmap / overlay).  
Resolution 300 dpi for IEEE publication.

In [ ]:
fig_cases = []
for label in CORE_LABELS:
    candidates = [m for m in all_metadata if m['label_name'] == label]
    if candidates:
        best = max(candidates, key=lambda x: x['confidence'])
        fig_cases.append(best)

n_rows = len(fig_cases)
fig, axes = plt.subplots(n_rows, 3, figsize=(10, n_rows * 2.8))

col_titles = ['Original', 'Grad-CAM (heatmap)', 'Sobreposição']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=11, fontweight='bold', pad=8)

for row, case in enumerate(fig_cases):
    stem       = Path(case['image_name']).stem
    label_slug = case['label_name'].replace('Ó','O').replace('Ú','U').replace('Ã','A').replace('É','E')

    orig_path    = IMGS_OUT / f'{stem}_{label_slug}_original.png'
    heatmap_path = IMGS_OUT / f'{stem}_{label_slug}_heatmap.png'
    overlay_path = IMGS_OUT / f'{stem}_{label_slug}_overlay.png'

    imgs_to_show = [orig_path, heatmap_path, overlay_path]

    for col, img_path in enumerate(imgs_to_show):
        ax = axes[row, col]
        if img_path.exists():
            ax.imshow(np.array(Image.open(img_path)))
        else:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

        if col == 0:
            conf = case['confidence']
            ax.set_ylabel(
                f"{case['label_name']}\nconf={conf:.2f}",
                fontsize=9, rotation=0, labelpad=60, va='center'
            )

plt.suptitle(
    'Grad-CAM — M2 Swin-Tiny λ=0.6\nMelhor predição por classe (fold-0 test)',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
fig.savefig(FIGS_DIR / 'gradcam_panel.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figura salva em:', FIGS_DIR / 'gradcam_panel.png')

## 9. Medical validation spreadsheet

Generates a CSV ready for the doctor to fill out.  
Columns: image, label, confidence, whether it has LIGHT, field for grade (1/2/3), field for shortcut (EROSION).

In [ ]:
rows_val = []
for entry in all_metadata:
    row_val = {
        'imagem':             entry['image_name'],
        'rotulo':             entry['label_name'],
        'confianca_modelo':   round(entry['confidence'], 3),
        'gt_correto':         'Sim' if entry['gt_label'] == 1 else 'Não',
        'tem_reflexo_luz':    'Sim' if entry['has_luz'] else 'Não',
        'arquivo_overlay':    entry['overlay_path'],
        'cam_mean':           round(entry['cam_stats']['mean'], 3),
        'cam_pct_acima_050':  round(entry['cam_stats']['pct_above_050'] * 100, 1),
        'cam_pct_acima_075':  round(entry['cam_stats']['pct_above_075'] * 100, 1),

        'concordancia_medica': '',
        'ativacao_na_lesao':   '',
        'observacao_medica':   '',
    }
    rows_val.append(row_val)

df_val_sheet = pd.DataFrame(rows_val)
df_val_sheet.to_csv(RESULTS_DIR / 'medical_validation_sheet.csv',
                    index=False, encoding='utf-8-sig')

print('=== PLANILHA DE VALIDAÇÃO MÉDICA ===')
print(df_val_sheet[['imagem', 'rotulo', 'confianca_modelo',
                     'tem_reflexo_luz', 'arquivo_overlay']].to_string())
print(f'\nSalva em: {RESULTS_DIR / "medical_validation_sheet.csv"}')
print('\nInstruções para a médica:')
print('  concordancia_medica : 1=Região errada / 2=Parcialmente correta / 3=Correta')
print('  ativacao_na_lesao   : S=Sim (lesão) / N=Não (artefato) — preencher só EROSÃO com LUZ')
print('  observacao_medica   : comentário livre')

## 10. Final summary and texts for the paper

In [ ]:
print('=' * 65)
print('RESUMO NB05 — GRAD-CAM E EXPLICABILIDADE CLÍNICA')
print('=' * 65)

print('\n--- Estatísticas Grad-CAM por classe ---')
print(f'{"Classe":<22} {"N":>4} {"Conf média":>12} {"CAM mean":>10} {"CAM std":>9} {"Top-10%":>8}')
print('-' * 70)
for label in CORE_LABELS:
    entries = [m for m in all_metadata if m['label_name'] == label]
    if not entries:
        continue
    conf_mean = np.mean([e['confidence'] for e in entries])
    cam_mean  = np.mean([e['cam_stats']['mean'] for e in entries])
    cam_std   = np.mean([e['cam_stats']['std']  for e in entries])
    top10     = np.mean([e['cam_stats']['top10_pct'] for e in entries])
    print(f'{label:<22} {len(entries):>4} {conf_mean:>12.3f} {cam_mean:>10.3f} {cam_std:>9.3f} {top10*100:>7.1f}%')

print('\n--- Shortcut LUZ→EROSÃO ---')
with open(RESULTS_DIR / 'shortcut_analysis.json', encoding='utf-8') as f:
    sc = json.load(f)
print(f'Co-ocorrência EROSÃO+LUZ no test : {sc["cooc_pct"]:.1%}')
print(f'F1 EROSÃO com LUZ                : {sc["f1_erosao_com_luz"]:.4f}')
print(f'F1 EROSÃO sem LUZ                : {sc["f1_erosao_sem_luz"]:.4f}')
print(f'Taxa FP EROSÃO quando só LUZ     : {sc["fp_rate_luz"]:.1%}')

if not isinstance(sc['f1_erosao_com_luz'], float) or not np.isnan(sc['f1_erosao_com_luz']):
    delta = sc['f1_erosao_com_luz'] - sc['f1_erosao_sem_luz']
    if abs(delta) > 0.05:
        direction = 'melhor com LUZ' if delta > 0 else 'pior com LUZ'
        print(f'Δ F1 (com vs sem LUZ)            : {delta:+.4f} → {direction}')
        if delta > 0.05:
            print('  → Possível shortcut: modelo usa LUZ como proxy para EROSÃO.')
        elif delta < -0.05:
            print('  → LUZ prejudica detecção de EROSÃO (confunde o modelo).')

print('\n--- Trecho sugerido para o artigo (seção Explicabilidade) ---')
n_total = len(all_metadata)
print(f"""
Para analisar o comportamento do modelo M2 (λ=0.6, Swin-Tiny),
foram gerados mapas Grad-CAM para {n_total} imagens de teste selecionadas
({N_IMGS_PER_CLASS} por classe, predições corretas com maior confiança, fold-0).
A camada alvo utilizada foi o último bloco da última stage do Swin-Tiny
(layers[-1].blocks[-1].norm1), com reshape_transform para conversão dos
patch tokens em representação espacial.

Para a classe EROSÃO, foi conduzida análise específica do potencial shortcut
LUZ→EROSÃO, quantificado pela co-ocorrência de {sc['cooc_pct']:.1%} entre as
anotações de reflexo de luz (LUZ) e erosão mucosa no conjunto de teste.
A taxa de falsos positivos de EROSÃO em imagens apenas com LUZ foi de
{sc['fp_rate_luz']:.1%}, indicando {'evidência de shortcut' if sc['fp_rate_luz'] > 0.2 else 'baixo impacto do artefato'}.

Os mapas de ativação foram avaliados por um gastroenterologista experiente
usando escala de concordância de 3 pontos (1=Errado, 2=Parcial, 3=Correto),
com análise específica da sobreposição entre ativação e artefato de reflexo
para os casos de EROSÃO com LUZ presente.
""")

print('\nOutputs gerados:')
print(f'  {RESULTS_DIR}/metadata.json')
print(f'  {RESULTS_DIR}/shortcut_analysis.json')
print(f'  {RESULTS_DIR}/medical_validation_sheet.csv')
print(f'  {FIGS_DIR}/gradcam_panel.png')
print(f'  {IMGS_OUT}/ ({n_total * 3} arquivos PNG)')